<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/CoronavirusTweetsNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
datatattle_covid_19_nlp_text_classification_path = kagglehub.dataset_download('datatattle/covid-19-nlp-text-classification')

print('Data source import complete.')


In [ ]:
pip install --upgrade torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 32.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 109.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 21.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 2.9 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import numpy as np
import random

def select_Seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)

select_Seed(42)

In [ ]:
!pip install transformers

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer,AutoModel
import os
from google.colab import userdata
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import gc

In [ ]:
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# !kaggle datasets download -d datatattle/covid-19-nlp-text-classification

In [ ]:
# !unzip -q /content/covid-19-nlp-text-classification.zip -d ./data_folder/

In [ ]:
train_df = pd.read_csv('/kaggle/input/datasets/datatattle/covid-19-nlp-text-classification/Corona_NLP_train.csv',encoding='latin-1')

In [ ]:
test_df = pd.read_csv('/kaggle/input/datasets/datatattle/covid-19-nlp-text-classification/Corona_NLP_test.csv')

In [ ]:
train_df

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3799,48751,London,16-03-2020,@MeNyrbie @Phil_Gahan @Chrisitv https://t.co/i...,Neutral
1,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
2,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
3,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
4,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative
...,...,...,...,...,...,...
41152,44951,89903,"Wellington City, New Zealand",14-04-2020,Airline pilots offering to stock supermarket s...,Neutral
41153,44952,89904,NaN,14-04-2020,Response to complaint not provided citing COVI...,Extremely Negative
41154,44953,89905,NaN,14-04-2020,You know itÂs getting tough when @KameronWild...,Positive
41155,44954,89906,NaN,14-04-2020,Is it wrong that the smell of hand sanitizer i...,Neutral


In [ ]:
autoToken = AutoTokenizer.from_pretrained('bert-base-uncased')
autoToken.vocab_size

In [ ]:
train_df = train_df[train_df['OriginalTweet'].str.strip() != ""]

In [ ]:
test_df = test_df[test_df['OriginalTweet'].str.strip() != ""]

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
train_label_encoder = LabelEncoder()
train_df['Sentiment'] = train_label_encoder.fit_transform(train_df['Sentiment'])

In [ ]:
test_label_encoder = LabelEncoder()
test_df['Sentiment'] = test_label_encoder.fit_transform(test_df['Sentiment'])

In [ ]:
print(len(train_label_encoder.classes_))

5


In [ ]:
train_statement = train_df['OriginalTweet'].astype(str).values
test_statement = test_df['OriginalTweet'].astype(str).values

In [ ]:
print(train_df['OriginalTweet'].str.split().str.len().mean())

30.5003037150424


In [ ]:
vocab_size = autoToken.vocab_size
max_len = 64

In [ ]:
train_tokenizer = autoToken(list(train_statement),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
test_tokenizer = autoToken(list(test_statement),padding='max_length',max_length=max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class EncodingStatement(Dataset):
  def __init__(self,encoding,labels):
    self.encoding = encoding
    self.labels = labels

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    input_ids = self.encoding['input_ids'][idx]
    attention_mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':input_ids,
        'attention_mask':attention_mask,
        'labels':torch.tensor(self.labels[idx],dtype=torch.long)
    }

In [ ]:
train_ds = EncodingStatement(encoding=train_tokenizer,labels=train_df['Sentiment'])
test_ds = EncodingStatement(encoding=test_tokenizer,labels=test_df['Sentiment'])

In [ ]:
train_loader = DataLoader(dataset=train_ds,batch_size=32,shuffle=True,pin_memory=True,num_workers=2)
test_loader = DataLoader(dataset=test_ds,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

In [ ]:
class BERTWritter(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.bert = AutoModel.from_pretrained('bert-base-uncased')
    self.dropout = nn.Dropout(0.2)
    self.outputlayer = nn.Linear(768,num_classes)

  def forward(self,input_ids,attention_mask):
    output = self.bert(input_ids=input_ids,attention_mask=attention_mask)
    x = output.last_hidden_state
    mask = attention_mask.unsqueeze(-1)
    mean = (x*mask).sum(dim=1) / mask.sum(dim=1)
    mean = self.dropout(mean)
    return self.outputlayer(mean)

In [ ]:
model = BERTWritter(num_classes=len(train_label_encoder.classes_))
if torch.cuda.device_count() > 1:
  model = nn.DataParallel(model)
model = model.to(device)

In [ ]:
optimiser = optim.AdamW(model.parameters(),lr=2e-5,weight_decay=0.01)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

In [ ]:
  gc.collect()
  torch.cuda.empty_cache()

In [ ]:
epochs = 5

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0
  progressbar_train = tqdm(train_loader,desc=f'Epoch: {epoch+1}')

  for batch in progressbar_train:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimiser.zero_grad()

    with torch.amp.autocast('cuda'):
      outputs = model(input_ids,attention_mask)
      loss = loss_fn(outputs,labels)

    scaler.scale(loss).backward()
    scaler.step(optimiser)
    scaler.update()

    train_loss += loss.item()
    progressbar_train.set_postfix(loss=loss.item())
    pred = torch.argmax(outputs,dim=1)
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  average_loss = train_loss / len(train_loader)
  train_accuracy = train_correct / train_total
  print(f'Train_accuracy: {train_accuracy:.4f} | Train_loss: {average_loss:.4f}')

  #_____Validation______#
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  progressbar_val = tqdm(test_loader,desc=f'Epoch: {epoch+1}')
  with torch.no_grad():
   for batch in progressbar_val:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    with torch.amp.autocast('cuda'):
      outputs = model(input_ids,attention_mask)
      loss = loss_fn(outputs,labels)

    val_loss += loss.item()
    progressbar_train.set_postfix(loss=loss.item())
    pred = torch.argmax(outputs,dim=1)
    val_correct += (pred == labels).sum().item()
    val_total += labels.size(0)

  average_loss = val_loss / len(test_loader)
  val_accuracy = val_correct / val_total
  print(f'Val_accuracy: {val_accuracy:.4f} | Val_loss: {average_loss:.4f}')

  gc.collect()
  torch.cuda.empty_cache()

Epoch: 1: 100%|██████████| 1287/1287 [03:53<00:00,  5.51it/s, loss=0.0966]


Train_accuracy: 0.8568 | Train_loss: 0.4011


Epoch: 1: 100%|██████████| 119/119 [00:08<00:00, 14.68it/s]


Val_accuracy: 0.8202 | Val_loss: 0.5285


Epoch: 2: 100%|██████████| 1287/1287 [03:53<00:00,  5.50it/s, loss=0.215] 


...

Val_accuracy: 0.8349 | Val_loss: 0.5770


Epoch: 5: 100%|██████████| 1287/1287 [03:54<00:00,  5.50it/s, loss=0.00141]


Train_accuracy: 0.9556 | Train_loss: 0.1300


Epoch: 5: 100%|██████████| 119/119 [00:08<00:00, 14.73it/s]


Val_accuracy: 0.8404 | Val_loss: 0.6096
